# Group Pipeline: Integrated Preprocessing
**Group:** 2026-Y2-S1-MLB-B4G2-05
**Dataset:** Tourism Recommendation Dataset-Tourist Satisfaction Prediction & Segmentation

This notebook combines all 6 members' finalized, individually-tested techniques into a single,
continuous pipeline. 

| Order | Section | Owner | Technique |
|---|---|---|---|
| 1 | Handling Missing Data | Fernando G.V.N. | Handling missing data |
| 2 | Feature Selection | Gimsara H.S.M.  | Feature engineering (selection) |
| 3 | Outlier Removal | Nirukshan R.  | Outlier removal |
| 4 | Encoding Categorical Variables | Wijayasinghe R.P.B.A.  | Encoding categorical variables |
| 5 | Normalization / Scaling | De Silva K.B.N. | Normalization / scaling |
| 6 | Dimension Reduction (PCA) | Wijetunga S.A.  | Feature engineering (dimension reduction) |


 Handling Missing Data

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv("tourism_recommendation_dataset_en.csv")
print("Shape:", df.shape)

Shape: (100000, 25)


In [3]:
n_dupes = df.duplicated().sum()
print(f"Duplicate rows found: {n_dupes}")
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

Duplicate rows found: 0
Shape after removing duplicates: (100000, 25)


Shape after removing duplicates: (100000, 25)


In [4]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)
missing_summary

,missing_count,missing_pct
group_fee,70242,70.24
trip_days,70242,70.24
main_spots,70242,70.24
transport_mode,70242,70.24


In [5]:
for col in ["group_fee", "trip_days", "transport_mode", "main_spots"]:
    print(col, "-> missing rate broken down by is_group_tour:")
    print(df.groupby("is_group_tour")[col].apply(lambda s: s.isnull().mean().round(3)))
    print()

group_fee -> missing rate broken down by is_group_tour:
is_group_tour
No     1.0
Yes    0.0
Name: group_fee, dtype: float64

trip_days -> missing rate broken down by is_group_tour:
is_group_tour
No     1.0
Yes    0.0
Name: trip_days, dtype: float64

transport_mode -> missing rate broken down by is_group_tour:
is_group_tour
No     1.0
Yes    0.0
Name: transport_mode, dtype: float64

main_spots -> missing rate broken down by is_group_tour:
is_group_tour
No     1.0
Yes    0.0
Name: main_spots, dtype: float64



In [6]:
df_clean = df.copy()
df_clean["is_group_tour_flag"] = (df_clean["is_group_tour"] == "Yes").astype(int)

for col in ["group_fee", "trip_days"]:
    group_median = df_clean.loc[df_clean["is_group_tour"] == "Yes", col].median()
    df_clean[col] = df_clean[col].fillna(group_median)

for col in ["transport_mode", "main_spots"]:
    df_clean[col] = df_clean[col].fillna("Not Applicable")

print("Remaining missing values:\n", df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Remaining missing values:
 Series([], dtype: int64)


In [7]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
assert df_clean.shape[0] == df.shape[0], "Row count should not change - we imputed, not dropped"

Original shape: (100000, 25)
Cleaned shape: (100000, 26)


Feature Engineering: Feature Selection 

In [8]:
# Continuing in-memory from Section 1's df_clean (no file read needed)
df = df_clean.copy()
sat_order = ["Neutral", "Satisfied", "Very Satisfied"]

In [9]:
leak_check = pd.crosstab(df["satisfaction_level"], df["recommendation_level"])
print(leak_check)
# recommendation_level maps 1:1 to satisfaction_level -> must be EXCLUDED (pure leakage)

recommendation_level  Highly Recommend  Neutral  Recommend
satisfaction_level                                        
Neutral                              0      241          0
Satisfied                            0        0      50518
Very Satisfied                   49241        0          0


In [10]:
print(df.groupby("satisfaction_level")["rating"].mean())
# rating is very strongly tied to satisfaction_level -> exclude from core feature set as well

satisfaction_level
Neutral           3.295021
Satisfied         4.140473
Very Satisfied    4.736468
Name: rating, dtype: float64


In [11]:
def chi_square_test(col):
    contingency = pd.crosstab(df[col], df["satisfaction_level"])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    return chi2, p_value

candidate_cats = ["gender", "age_group", "season", "is_holiday", "is_group_tour",
                   "attraction_level", "attraction_category", "province", "source_province"]

chi2_results = pd.DataFrame(
    {col: chi_square_test(col) for col in candidate_cats},
    index=["chi2_statistic", "p_value"]
).T.sort_values("chi2_statistic", ascending=False)

chi2_results

,chi2_statistic,p_value
attraction_level,23446.205580,0.000000
attraction_category,16317.407977,0.000000
province,3015.586358,0.000000
source_province,78.435801,0.140464
age_group,8.823446,0.357408
is_holiday,5.835028,0.054068
gender,3.735630,0.154461
season,2.631491,0.853471
is_group_tour,0.223552,0.894244


In [12]:
df["sat_ord"] = df["satisfaction_level"].map({"Neutral": 0, "Satisfied": 1, "Very Satisfied": 2})
numeric_candidates = ["age", "ticket_price", "visit_duration_hours", "spend_amount", "other_spend"]
corr_scores = df[numeric_candidates].corrwith(df["sat_ord"]).sort_values(key=abs, ascending=False)
corr_scores

ticket_price            0.138148
spend_amount            0.057485
other_spend             0.006335
age                    -0.004114
visit_duration_hours   -0.001450
dtype: float64

In [13]:
selected_features = [
    "attraction_level", "attraction_category", "province", "ticket_price", "spend_amount",
]
excluded_leakage = ["recommendation_level", "rating", "tourist_id"]
excluded_weak = ["gender", "age", "age_group", "season", "is_holiday", "is_group_tour",
                  "visit_duration_hours", "other_spend", "source_province"]

print("Selected:", selected_features)
print("\nExcluded (leakage):", excluded_leakage)
print("\nExcluded (negligible signal per chi-square/correlation):", excluded_weak)

Selected: ['attraction_level', 'attraction_category', 'province', 'ticket_price', 'spend_amount']

Excluded (leakage): ['recommendation_level', 'rating', 'tourist_id']

Excluded (negligible signal per chi-square/correlation): ['gender', 'age', 'age_group', 'season', 'is_holiday', 'is_group_tour', 'visit_duration_hours', 'other_spend', 'source_province']


In [14]:
df_selected = df[selected_features + ["satisfaction_level"]].copy()
print("Shape after feature selection:", df_selected.shape)

Shape after feature selection: (100000, 6)


Outlier Removal

In [15]:
df = df_selected.copy()
numeric_cols = ["ticket_price", "spend_amount"]
df[numeric_cols].describe()

,ticket_price,spend_amount
count,100000.000000,100000.000000
mean,80.783510,255.433621
std,67.183866,179.814679
min,0.000000,0.000000
25%,35.000000,98.000000
50%,70.000000,227.200000
75%,115.000000,401.090000
max,399.000000,898.220000


In [16]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper

outlier_report = {}
for col in numeric_cols:
    lower, upper = iqr_bounds(df[col])
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = {"lower_bound": round(lower, 2), "upper_bound": round(upper, 2),
                            "n_outliers": n_outliers, "pct_outliers": round(n_outliers / len(df) * 100, 2)}

pd.DataFrame(outlier_report).T

,lower_bound,upper_bound,n_outliers,pct_outliers
ticket_price,-85.00,235.00,2888.0,2.89
spend_amount,-356.64,855.72,27.0,0.03


In [17]:
df_clean = df.copy()
for col in numeric_cols:
    lower, upper = iqr_bounds(df[col])
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)

print("Before capping - spend_amount max:", df["spend_amount"].max())
print("After capping  - spend_amount max:", df_clean["spend_amount"].max())

Before capping - spend_amount max: 898.22
After capping  - spend_amount max: 855.7249999999999


Encoding Categorical Variables

In [18]:
# Continuing from Section 3's df_clean (outlier-capped output)
df = df_clean.copy()
cat_cols = ["attraction_category", "attraction_level", "province"]
cardinality = df[cat_cols].nunique().sort_values()
cardinality

attraction_level        3
province               22
attraction_category    26
dtype: int64

In [19]:
# Label Encoding - attraction_level is ordinal (3A < 4A < 5A)
df_encoded = df.copy()
level_order = {"3A": 0, "4A": 1, "5A": 2}
df_encoded["attraction_level_encoded"] = df_encoded["attraction_level"].map(level_order)
df_encoded = df_encoded.drop(columns=["attraction_level"])
df_encoded[["attraction_level_encoded"]].drop_duplicates().sort_values("attraction_level_encoded")

,attraction_level_encoded
87,0
0,1
2,2


In [20]:
# One-Hot Encoding - attraction_category is nominal
df_encoded = pd.get_dummies(df_encoded, columns=["attraction_category"], drop_first=True)
print("Shape after one-hot encoding attraction_category:", df_encoded.shape)

Shape after one-hot encoding attraction_category: (100000, 30)


In [21]:
# Target Encoding - province is high-cardinality
sat_numeric = df["satisfaction_level"].map({"Neutral": 0, "Satisfied": 1, "Very Satisfied": 2})
province_target_means = sat_numeric.groupby(df["province"]).mean()
df_encoded["province_target_enc"] = df["province"].map(province_target_means)

print(province_target_means.sort_values(ascending=False).head())
df_encoded[["province_target_enc"]].drop_duplicates().head()

province
Zhejiang     1.659115
Guangdong    1.580707
Jiangsu      1.577940
Hunan        1.535797
Hubei        1.531401
Name: satisfaction_level, dtype: float64


,province_target_enc
0,1.427281
1,1.482789
2,1.577940
3,1.379714
4,1.519129


In [22]:
df_encoded = df_encoded.drop(columns=["province"])
print("Final shape after full encoding:", df_encoded.shape)

Final shape after full encoding: (100000, 30)


Normalization / Scaling

In [23]:
df = df_encoded.copy()
numeric_cols = ["ticket_price", "spend_amount"]
df[numeric_cols].describe()

,ticket_price,spend_amount
count,100000.000000,100000.000000
mean,78.562800,255.428265
std,59.894242,179.796409
min,0.000000,0.000000
25%,35.000000,98.000000
50%,70.000000,227.200000
75%,115.000000,401.090000
max,235.000000,855.725000


In [24]:
ranges = df[numeric_cols].agg(["min", "max"]).T
ranges["range"] = ranges["max"] - ranges["min"]
ranges

,min,max,range
ticket_price,0.0,235.000,235.000
spend_amount,0.0,855.725,855.725


In [25]:
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df[numeric_cols])
df_scaled[numeric_cols].describe().round(2)

,ticket_price,spend_amount
count,100000.00,100000.00
mean,0.00,0.00
std,1.00,1.00
min,-1.31,-1.42
25%,-0.73,-0.88
50%,-0.14,-0.16
75%,0.61,0.81
max,2.61,3.34


Feature Engineering: Dimension Reduction / PCA 

In [26]:
# Continuing from Section 5's df_scaled (final teammate's step)
df = df_scaled.copy()
print("Shape entering PCA:", df.shape)

Shape entering PCA: (100000, 30)


In [27]:
target = df["satisfaction_level"]
features_for_pca = df.drop(columns=["satisfaction_level"])
features_for_pca.head()

,ticket_price,spend_amount,attraction_level_encoded,attraction_category_Botanical Garden,attraction_category_Cheng Shi Gong Yuan,attraction_category_Food Street,attraction_category_Gong Ye Lv You,attraction_category_Historical Architecture,attraction_category_Historical Culture,attraction_category_Historical Site,...,attraction_category_Theme Park,attraction_category_Urban Landmark,attraction_category_Urban Landscape,attraction_category_Wen Bo Yuan Guan,attraction_category_Wen Hua Lv You,attraction_category_Wen Hua Yi Shu,attraction_category_Zhu Ti Zhan Lan,attraction_category_Zi Ran Qi Guan,attraction_category_Zoo,province_target_enc
0,0.775324,0.590069,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1.427281
1,-0.226447,-1.059138,1,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,1.482789
2,0.357919,1.189361,2,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1.577940
3,-0.309928,-1.086947,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1.379714
4,1.359690,-0.530760,2,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1.519129


In [28]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(features_for_pca),
    columns=features_for_pca.columns
)
print("Standardized data (mean ~0, std ~1):")
X_scaled.describe().round(2)

Standardized data (mean ~0, std ~1):


,ticket_price,spend_amount,attraction_level_encoded,attraction_category_Botanical Garden,attraction_category_Cheng Shi Gong Yuan,attraction_category_Food Street,attraction_category_Gong Ye Lv You,attraction_category_Historical Architecture,attraction_category_Historical Culture,attraction_category_Historical Site,...,attraction_category_Theme Park,attraction_category_Urban Landmark,attraction_category_Urban Landscape,attraction_category_Wen Bo Yuan Guan,attraction_category_Wen Hua Lv You,attraction_category_Wen Hua Yi Shu,attraction_category_Zhu Ti Zhan Lan,attraction_category_Zi Ran Qi Guan,attraction_category_Zoo,province_target_enc
count,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,...,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00
mean,0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,-0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,-0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.31,-1.42,-2.73,-0.11,-0.12,-0.05,-0.08,-0.23,-0.38,-0.18,...,-0.27,-0.08,-0.08,-0.13,-0.05,-0.09,-0.05,-0.14,-0.13,-1.73
25%,-0.73,-0.88,-0.84,-0.11,-0.12,-0.05,-0.08,-0.23,-0.38,-0.18,...,-0.27,-0.08,-0.08,-0.13,-0.05,-0.09,-0.05,-0.14,-0.13,-0.74
50%,-0.14,-0.16,-0.84,-0.11,-0.12,-0.05,-0.08,-0.23,-0.38,-0.18,...,-0.27,-0.08,-0.08,-0.13,-0.05,-0.09,-0.05,-0.14,-0.13,-0.11
75%,0.61,0.81,1.05,-0.11,-0.12,-0.05,-0.08,-0.23,-0.38,-0.18,...,-0.27,-0.08,-0.08,-0.13,-0.05,-0.09,-0.05,-0.14,-0.13,0.58
max,2.61,3.34,1.05,9.14,8.39,21.59,11.86,4.44,2.63,5.70,...,3.66,11.85,11.83,7.79,20.74,11.65,21.15,6.92,7.85,2.65


In [29]:
pca_full = PCA()
pca_full.fit(X_scaled)

eigenvalues = pca_full.explained_variance_
explained_var_pct = pca_full.explained_variance_ratio_ * 100
cumulative_pct = np.cumsum(explained_var_pct)

eigen_table = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(eigenvalues))],
    "eigenvalue": eigenvalues.round(3),
    "explained_variance_%": explained_var_pct.round(1),
    "cumulative_%": cumulative_pct.round(1)
})
eigen_table

,component,eigenvalue,explained_variance_%,cumulative_%
0,PC1,2.052,7.1,7.1
1,PC2,1.455,5.0,12.1
2,PC3,1.359,4.7,16.8
3,PC4,1.116,3.8,20.6
4,PC5,1.102,3.8,24.4
5,PC6,1.066,3.7,28.1
6,PC7,1.051,3.6,31.7
7,PC8,1.041,3.6,35.3
8,PC9,1.031,3.6,38.9
9,PC10,1.026,3.5,42.4


In [30]:
eigenvectors = pd.DataFrame(
    pca_full.components_,
    columns=features_for_pca.columns,
    index=[f"PC{i+1}" for i in range(len(eigenvalues))]
)
eigenvectors.round(2)

,ticket_price,spend_amount,attraction_level_encoded,attraction_category_Botanical Garden,attraction_category_Cheng Shi Gong Yuan,attraction_category_Food Street,attraction_category_Gong Ye Lv You,attraction_category_Historical Architecture,attraction_category_Historical Culture,attraction_category_Historical Site,...,attraction_category_Theme Park,attraction_category_Urban Landmark,attraction_category_Urban Landscape,attraction_category_Wen Bo Yuan Guan,attraction_category_Wen Hua Lv You,attraction_category_Wen Hua Yi Shu,attraction_category_Zhu Ti Zhan Lan,attraction_category_Zi Ran Qi Guan,attraction_category_Zoo,province_target_enc
PC1,0.60,0.38,0.42,-0.08,-0.14,-0.10,-0.04,-0.06,-0.13,-0.01,...,0.34,0.08,-0.09,-0.17,0.01,-0.12,0.03,-0.01,-0.04,0.23
PC2,-0.18,-0.18,0.24,-0.11,-0.03,-0.08,-0.06,0.02,-0.09,-0.12,...,-0.26,-0.19,-0.01,-0.05,0.03,-0.12,-0.17,-0.11,-0.23,0.50
PC3,0.11,0.10,-0.28,0.02,-0.04,0.08,0.02,-0.18,-0.35,-0.05,...,-0.04,0.08,-0.05,-0.01,-0.03,0.04,0.10,-0.03,0.10,-0.24
PC4,0.02,0.01,0.10,-0.07,-0.14,-0.13,-0.11,-0.29,0.79,-0.04,...,-0.13,0.03,-0.06,-0.19,-0.11,-0.05,0.02,-0.04,-0.03,-0.09
PC5,-0.01,0.08,-0.25,-0.04,0.06,0.24,0.18,0.17,0.23,-0.20,...,0.46,-0.10,-0.06,0.18,0.27,-0.07,-0.07,-0.13,-0.10,0.23
PC6,0.02,-0.00,-0.06,0.01,-0.00,0.09,0.05,-0.16,0.08,-0.07,...,-0.18,0.03,-0.03,0.04,0.05,0.01,0.04,-0.04,0.04,0.01
PC7,0.02,-0.02,-0.00,-0.02,-0.05,-0.01,0.00,0.73,0.02,-0.02,...,-0.18,0.02,-0.03,-0.04,0.03,-0.04,0.00,-0.01,-0.02,0.01
PC8,-0.01,0.00,-0.07,0.02,0.15,0.19,0.14,-0.43,-0.01,-0.27,...,0.06,-0.11,0.02,0.28,0.17,0.01,-0.06,-0.09,-0.06,0.05
PC9,-0.02,-0.00,0.02,0.00,0.15,-0.04,-0.03,-0.16,-0.04,0.79,...,0.16,-0.17,0.10,0.13,-0.01,0.02,-0.12,-0.00,-0.18,0.01
PC10,0.03,0.01,-0.06,-0.11,-0.22,0.20,0.23,-0.02,0.04,0.42,...,-0.32,0.18,-0.23,-0.02,0.30,-0.20,0.08,-0.30,0.01,0.05


In [31]:
for pc in eigenvectors.index[:2]:
    top_features = eigenvectors.loc[pc].abs().sort_values(ascending=False)
    print(f"{pc} - features ranked by loading magnitude:")
    print(top_features.round(2))
    print()

PC1 - features ranked by loading magnitude:
ticket_price                                   0.60
attraction_level_encoded                       0.42
spend_amount                                   0.38
attraction_category_Theme Park                 0.34
province_target_enc                            0.23
attraction_category_Wen Bo Yuan Guan           0.17
attraction_category_Cheng Shi Gong Yuan        0.14
attraction_category_Historical Culture         0.13
attraction_category_Wen Hua Yi Shu             0.12
attraction_category_Food Street                0.10
attraction_category_Urban Landscape            0.09
attraction_category_Natural Culture            0.09
attraction_category_Revolutionary Site         0.09
attraction_category_Urban Landmark             0.08
attraction_category_Botanical Garden           0.08
attraction_category_Religious Culture          0.07
attraction_category_Sports & Leisure           0.06
attraction_category_Min Su Wen Hua             0.06
attraction_category_

In [32]:
n_components_90 = np.argmax(cumulative_pct >= 90) + 1
print(f"Components needed to retain 90% of variance: {n_components_90} (out of {X_scaled.shape[1]} original features)")

pca_final = PCA(n_components=n_components_90)
X_reduced = pca_final.fit_transform(X_scaled)
print("Reduced dataset shape:", X_reduced.shape)

Components needed to retain 90% of variance: 24 (out of 29 original features)
Reduced dataset shape: (100000, 24)


## Final Output

In [33]:
import os

final_df = pd.DataFrame(X_reduced, columns=[f"PC{i+1}" for i in range(n_components_90)])
final_df["satisfaction_level"] = target.values

output_dir = "results/outputs"
os.makedirs(output_dir, exist_ok=True)
output_path = f"{output_dir}/processed_tourism_dataset.csv"
final_df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")
print("Final model-ready dataset shape:", final_df.shape)
final_df.head()

Saved to results/outputs/processed_tourism_dataset.csv
Final model-ready dataset shape: (100000, 25)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23,PC24,satisfaction_level
0,0.421084,-1.309662,-0.354689,-0.159513,-1.963282,-1.776185,1.216489,3.100193,-0.419602,0.643121,...,-0.260233,-0.239338,-0.136154,-0.080446,-0.202149,-0.041775,0.080752,-0.118303,-0.036255,Satisfied
1,-1.332975,-0.409108,-1.088941,2.226674,0.808738,0.269927,0.069983,0.011668,-0.138417,0.172170,...,-0.031940,-0.047592,-0.036120,-0.013725,-0.048147,0.005879,0.010551,-0.015387,-0.005119,Very Satisfied
2,1.422851,0.532844,-0.630894,-0.054753,0.143908,-0.049085,-0.006435,0.013128,0.026920,0.027160,...,0.014621,0.005518,0.019897,-0.003082,0.026151,-0.020085,-0.016595,0.011112,0.006278,Satisfied
3,-1.364300,-0.925003,0.340326,-0.006347,-0.258674,0.016008,0.002526,-0.034372,-0.018134,-0.044673,...,-0.003458,0.008026,-0.012596,0.006764,-0.012811,0.021307,0.013497,-0.005577,-0.003147,Satisfied
4,1.511910,0.353873,-1.623823,-0.982954,-1.096606,3.035668,-0.192669,0.119947,-0.247967,-0.045276,...,-0.155742,-0.138016,-0.099486,-0.007574,-0.137054,-0.013810,0.035066,-0.084232,-0.024461,Very Satisfied
